# Задание 10: Fasttext, idf + fasttext

В работе реализуется система поиска с использованием усреднённых векторов слов, полученных с помощью fasttext, и взвешенных векторов с использованием TF-IDF.

In [4]:
import numpy as np
import fasttext
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f if line.strip()]

queries = [
    "Памятник на месте рождения Пушкина стоит не там.",
    "Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле.",
    "Победный гол сальвадорского футболиста (на илл.) в ворота соперника привёл к шестидневной войне."
]

Используем модель fasttext с предобученными векторами "wiki.ru.vec". Файл был загружен по ссылке https://fasttext.cc/docs/en/pretrained-vectors.html

In [2]:
from gensim.models import KeyedVectors

fasttext_path = "/home/aigul/Downloads/wiki.ru/wiki.ru.vec"
ft_model = KeyedVectors.load_word2vec_format(fasttext_path, binary=False)

Создадим TF-IDF векторизатор, обучим его на корпусе и запросах и сформируем словарь слов с соответствующими значениями IDF:

In [7]:
# TF-IDF
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus + queries)
idf_dict = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))

Сформируем матрицы векторов для всего корпуса с использованием и без использования IDF:

In [11]:
def sentence_vector(sentence, use_idf=False):
    words = [w.lower() for w in sentence.split() if w.lower() in ft_model.key_to_index]
    if not words:
        return np.zeros(ft_model.vector_size)

    vectors = []
    for w in words:
        weight = idf_dict.get(w, 1.0) if use_idf else 1.0
        vectors.append(ft_model[w] * weight)
    return np.mean(vectors, axis=0)

corpus_vecs = np.array([sentence_vector(sent, use_idf=False) for sent in corpus])
corpus_vecs_idf = np.array([sentence_vector(sent, use_idf=True) for sent in corpus])

In [16]:
from razdel import sentenize
import numpy as np

# разбиваем текст корпуса на отдельные предложения
corpus_sentences = []
for doc in corpus:
    corpus_sentences.extend([sent.text for sent in sentenize(doc)])

def sentence_vector(sentence, use_idf=False):
    words = [w.lower() for w in sentence.split() if w.lower() in ft_model.key_to_index]
    if not words:
        return np.zeros(ft_model.vector_size)

    vectors = []
    for w in words:
        weight = idf_dict.get(w, 1.0) if use_idf else 1.0
        vectors.append(ft_model[w] * weight)
    return np.mean(vectors, axis=0)

corpus_vecs = np.array([sentence_vector(sent, use_idf=False) for sent in corpus_sentences])
corpus_vecs_idf = np.array([sentence_vector(sent, use_idf=True) for sent in corpus_sentences])


In [19]:
# поиск релевантных предложений
for q in queries:
    q_vec = sentence_vector(q, use_idf=False).reshape(1, -1)
    q_vec_idf = sentence_vector(q, use_idf=True).reshape(1, -1)

    sim = cosine_similarity(q_vec, corpus_vecs)[0]
    sim_idf = cosine_similarity(q_vec_idf, corpus_vecs_idf)[0]

    top_k = 10
    top_idx = np.argsort(sim)[::-1][:top_k]
    top_idx_idf = np.argsort(sim_idf)[::-1][:top_k]

    print("=================================================================")
    print(f"Запрос: {q}")
    print("\nУсреднённые векторы fasttext:")
    for i in top_idx:
        print(f"  ({sim[i]:.3f}) {corpus_sentences[i]}")
    print("\nУсреднение с учётом IDF:")
    for i in top_idx_idf:
        print(f"  ({sim_idf[i]:.3f}) {corpus_sentences[i]}")

Запрос: Памятник на месте рождения Пушкина стоит не там.

Усреднённые векторы fasttext:
  (0.842) Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
  (0.790) Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
  (0.741) Отличие памятника от многочисленных других памятников Пушкину состоит в том, что он изображает поэта в юном возрасте.
  (0.737) Памятник был сооружён вблизи того места, где ранее находился дом по Немецкой (ныне — Бауманская) улице, в котором, как предполагалось, 26 мая (6 июня) 1799 года родился русский поэт и писатель Александр Сергеевич Пушкин (1799—1837).
  (0.695) На стене соседней школы находится мемориальная доска из красного гранита с бронзовым барельефом Пушкина, созданная скульптором Константином Кошкиным.
  (0.675) Дере

## Выводы:

1. В целом модели выдавали примерно одинаковые результаты. Особенно ответы на первых местах. Скорее всего это обусловлено маленьким размером корпуса. 
2. Иногда использование idf помогает выдавать более содержательные ответы (за счет придания редким словам большего веса). Например, в запросе "Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле"  обычный fasttext выдал на 7 месте предложение совсем из другой статьи про "футбольную войну", а с idf это предложение сместилось на 10 место. Но можно видеть и минус: нужный факт про "кукурузное поле" оказалася ближе к началу в модели только с fasttext, так как модель с idf посчитала это предложение неинформативным из-за более обычных слов. 
В третьем запросе тоже можно наблюдать, что использование idf иногда позволяет вывести вперед предложения с подтверждающими фактами.